# Bangladesh: 0223_climate_bootcamp_session7

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0223_climate_bootcamp_session7
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'climate' / '0223_climate_bootcamp_session7'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""07_class7_temp_downscaling.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Climate Data Analysis Bootcamp using GEE and Python - Session7: ERA5 Temperature Downscaling
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Image Collection**'
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate('2015', '2026').select('temperature_2m')
era5
time_start = ee.Date('2015')
time_end = ee.Date('2026')
time_dif = time_end.difference(time_start, 'month')
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(lambda x: time_start.advance(x, 'month'))
time_list

def monthly(date_list, col):
    start_month = ee.Date(date_list)
    end_month = start_month.advance(1, 'month')
    col_month = col.filterDate(start_month, end_month).mean()
    return col_month.set('system:time_start', start_month.millis())
ndvi = ee.ImageCollection('MODIS/061/MOD13A2').filterDate(time_start, time_end).select('EVI', 'NDVI')
lst = ee.ImageCollection('MODIS/061/MOD11A2').filterDate(time_start, time_end).select('LST_Day_1km', 'LST_Night_1km')
et = ee.ImageCollection('MODIS/061/MOD16A2GF').filterDate(time_start, time_end).select('ET', 'PET')
lc = ee.ImageCollection('MODIS/061/MCD12Q1').filterDate(time_start, time_end).select('LC_Type1').mode()
topo = ee.Image('USGS/GTOPO30')
ndvi_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, ndvi)))
ndvi_monthly
lst_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, lst)))
lst_monthly
et_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, et)))
et_monthly
collection = ndvi_monthly.combine(lst_monthly).combine(et_monthly).map(lambda img: img.addBands(lc).addBands(topo))
collection
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_10km = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.1, -0.1))
pred_10km = xr.open_dataset(collection, engine='ee', **grid_10km)
pred_10km = pred_10km.sortby('time') * 1
temp_10km = xr.open_dataset(era5, engine='ee', **grid_10km)
temp_10km = temp_10km.sortby('time') - 273.15
ds_10km = xr.merge([pred_10km, temp_10km])
ds_10km
'# **Sampling**'
df_10km = ds_10km.to_dataframe()
df_10km
x = df_10km.drop('temperature_2m', axis=1)
x
y = df_10km['temperature_2m']
y
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse
r2 = r2_score(y_test, y_pred)
r2
'# **Downscaling**'
grid_1km = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.01, -0.01))
ds_1km = xr.open_dataset(collection, engine='ee', **grid_1km)
ds_1km = ds_1km.sortby('time') * 1
df_1km = ds_1km.to_dataframe()
df_1km['temp_pre'] = model.predict(df_1km)
dx = df_1km.to_xarray()
'# **Visualization**'
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ds_10km.temperature_2m.isel(time=103).plot(ax=ax[0], x='x', y='y', robust=True, cmap='turbo')
dx.temp_pre.isel(time=103).plot(ax=ax[1], x='x', y='y', robust=True, cmap='turbo')
plt.tight_layout()


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'climate', '0223_climate_bootcamp_session7', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'climate')  # explicit upload; requires Drive OAuth scope
